# Analysis

## Setting parameters

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from hhelper.tester import calculate_score, count_all_rules, count_all_attributes
from hhelper.typesetting import bold
from pathlib import Path
from hhelper.scoring import balanced_accuracy_score
import pandas as pd
import numpy as np

In [194]:
data_sets = [  # actual set for QR
    'australian',
     # 'automobile', # cat feats
     'bands',
     'bupa',
     'cleveland',
     # 'contraceptive', # no rules found
     # 'crx', # cat feats
     'dermatology',
     'ecoli',
     # 'german', # cat feats
     'glass', 
     # 'haberman', # no rules found
     'heart',
     'ionosphere',
     # 'mammographic',  # no rules found
     'pima',
     # 'saheart', # cat feats
     'sonar',  # 70 features!
     'spectfheart',
     'vehicle',
     'vowel',
     'wine',
     # 'winequality-red',  # todo temp uit voor inclusion tests
     'wisconsin',
     'yeast'
]

def_sets = [
    'australian',
    'bupa',
    'cleveland',
    # 'crx',
    # 'dermatology',
    'ecoli',
    'glass',
    # 'haberman',
    'heart',
    'ionosphere',
    'pima',
    # 'sonar',
    'spectfheart',
    # 'saheart',
    # 'wdbc',
    'wine',
    'winequality-red',
    'wisconsin',
    'yeast',
    # 'automobile',
    # 'dermatology', run base again
    # 'german',
    # 'movement',
    # 'vehicle',
    # 'vowel',
]
inclusion_test = [
    # 'ecoli',
    # 'wisconsin',
    # 'dermatology',
    # 'cleveland',
    # 'spectfheart',
    # 'bupa',
    # 'yeast',
    # 'heart',
    # 'australian',
    # 'glass',
    # 'pima',
    'australian',
     'bands',
     'bupa',
     'cleveland',
     # 'dermatology', temp 01/07/24
     'ecoli',
     'glass',   # todo tijdelijk bij prior-incl
     'heart',
     'ionosphere',
     'pima',
     'sonar',  # 70 features! eventjes skip voor QuickReduct 25/04/2024
     'spectfheart',
     'vehicle',
     # 'vowel',
     'wine',
     # 'winequality-red',  # temp uit voor inclusion tests
     'wisconsin',
     'yeast'  # todo tijdelijk bij prior-incl
]

qr_list = [
    'australian',
    'bupa',
    'cleveland',
    'dermatology',
    'ecoli',
    'glass',
    'heart',
    'pima',
    'spectfheart',
    # 'vehicle',
    # 'vowel',
    'wisconsin',
    'yeast'
]

In [195]:
data_sets = inclusion_test

In [196]:
len(data_sets)

15

In [197]:
data_base = Path.cwd() / 'keel-data'
metric = balanced_accuracy_score  # balanced_
scores = {}
rules = {}
attributes = {}
results_base = Path.cwd() / 'results'

## Loading WEKA results
MODLEM, FURIA, RIPPER

In [198]:
weka_folder = Path.cwd() / 'weka-results'
weka_models = {
    'modlem':'&',
    'furia':'and',
    'ripper':'and'
}
acc_type = 'balanced-accuracy'  # 'weka-accuracy.csv'

In [199]:
for name, connection in weka_models.items():
    model_folder = weka_folder / f"{name}-files"
    file = model_folder / f"{name}-{acc_type}.csv"
    scores[name] = pd.read_csv(weka_folder / file, header=None, index_col=0).to_dict()[1]
    rules[name] = {}
    attributes[name] = {}
    for file in model_folder.iterdir():
        if file.name[-3:] == 'txt':
            short_name = file.name[:-4]
            with open(file, 'r') as f:
                line = f.readline()
                nrs = []
                while len(line) > 4:
                    nrs.append(line.count(connection) + 1)
                    line = f.readline()
            rules[name][short_name] = len(nrs)
            attributes[name][short_name] = np.average(nrs)

## Loading QuickRules results

In [200]:
qr_models = {
    'qr': results_base / 'no-prune-results' / 'close-min-max-combo-results',
    # 'lin-owa': results_base / 'no-prune-results' / 'rules-lin-owa-qr-combo-minmax-results',
    # 'trun-lin-owa': results_base / 'no-prune-results' / 'rules-trun-lin-owa-qr-combo-minmax-results',
    # 'trun-exp-owa': results_base / 'no-prune-results' / 'trun-exp-owa-qr-combo-minmax-results',
    # 'avg-qr': results_base / 'mon-avg-std-minmax-results-2',
    # 'avg-lin-owa' : results_base / 'mon-avg-lin-owa-minmax-results-2',
    # 'prune-qr': results_base / 'prune-results' / 'prune-qr-combo-minmax-results',
    # 'prune-lin-owa': results_base / 'prune-results' / 'prune-lin-owa-qr-combo-minmax-results',
    # 'prune-trun-lin-owa': results_base / 'prune-results' / 'prune-trun-lin-owa-qr-combo-min-max-results',
    # 'prune-trun-exp-owa': results_base / 'prune-results' / 'prune-trun-exp-owa-qr-combo-min-max-results',
    # 'prune-avg-qr': results_base / 'prune-mon-avg-std-minmax-results-2',
    # 'prune-avg-lin-owa' : results_base / 'prune-mon-avg-lin-owa-minmax-results-2',
}

In [201]:
for model, path in qr_models.items():
    scores[model] = calculate_score(
        data_folder=data_base,
        results_folder=path,
        include=data_sets,
        metric=metric
    )
    rules[model] = count_all_rules(
        Path.cwd() / results_base / path,
        include=data_sets
    )
    attributes[model] = count_all_attributes(
        Path.cwd() / results_base / path,
        include=data_sets
    )

## Adding results for non rule based models


In [202]:
other_models =  {
    'naive-bayes': Path.cwd() / 'NaiveBayes-results',
    'svm': Path.cwd() / 'svm',
    'tree': Path.cwd() / 'decision-tree'
}

In [203]:
for model, path in other_models.items():
    print(model)
    scores[model] = calculate_score(
        data_folder=data_base,
        results_folder=path,
        include=data_sets,
        metric=metric,
        verbose=False
    )

naive-bayes
svm
tree


In [204]:
frnn_results = pd.read_csv(Path.cwd() / 'frnn_new_results.csv', header=None, index_col=0).to_dict()[1]
scores['frnn'] =  {data_set: frnn_results[data_set] for data_set in data_sets}

## FRRI Models

In [272]:
frri_models = {
    # 'non-overlap': Path.cwd() / 'non-overlap-rules',
    # 'nori' : Path.cwd() / 'non-overlap-rules-def_sets',
    # 'lower-nori' : Path.cwd() / 'lower-approx-rules-def_sets',
    # 'lower-check' : Path.cwd() / 'lower-approx-og-implement-test',
    # 'lower-new-check' : Path.cwd() / results_base / 'lower-approx-new-impl-test',
    # 'lower-incl-t1': Path.cwd() / results_base / 'lower-approx-luka-impl-incl.99'
    # 'incl 1e-6' : Path.cwd() / results_base / 'lower-approx' / 'inclusion' / "1-1e-6",
    'incl 1e-4' : Path.cwd() / results_base / 'lower-approx-new-impl-test',
    # 'incl 1e-3' : Path.cwd() / results_base / 'lower-approx' / 'inclusion' / "999",
    # 'incl 1e-2' : Path.cwd() / results_base / 'lower-approx' / 'inclusion' / "99",
    # 'incl 0.95' : Path.cwd() / results_base / 'lower-approx' / 'inclusion' / "95",
    # 'owa 1e-6' : Path.cwd() / results_base / 'lower-approx' / 'linear-owa-inclusion' / '1-1e-6',
    # 'quickreduct 1e-6': Path.cwd() / results_base / 'lower-approx' / 'quickreduct-ordering-bug' / '1-1e-6',
    # 'cv-thres': Path.cwd() / results_base / 'lower-approx' / 'cv-inclusion',
    # 'frfs': Path.cwd() / results_base / 'lower-approx' / 'frfs',
    # 'msecvx-no-reducts': Path.cwd() / results_base / 'multiclassMSECVX' / 'no-reducts',
    # 'maecvx-no-reducts': Path.cwd() / results_base / 'multiclassMAECVX' / 'no-reducts',
    # 'msecvx-0.5' : Path.cwd() / 'results' / 'multiclassMSECVX' / 'no-reducts-0.5-threshold',
    # 'maecvx-0.5-reducts' : Path.cwd() / 'results' / 'multiclassMAECVX' / 'reducts-0.5-threshold',
    # 'msecvx-0.5-reducts': Path.cwd() / 'results' / 'multiclassMSECVX' / 'reducts-0.5-threshold',
    # 'msecvx-i95-reducts': Path.cwd() / 'results' / 'multiclassMSECVX' / 'reducts-incl95-cov05',
    'msecvx-i50-reducts': Path.cwd() / 'results' / 'multiclassMSECVX' / 'reducts-incl5-cov05',
    'relabel-msecvx-reducts-default': Path.cwd() / 'results' / 'multiclassMSECVX' / 'relabelling-reducts-default',
    'relabel-maecvx-reducts-default': Path.cwd() / 'results' / 'multiclassMAECVX' / 'relabelling-reducts-default',
    'relabel-msecvx-c50-reducts': Path.cwd() / 'results' / 'multiclassMSECVX' / 'relabelling-reducts-cov50',
    'relabel-maecvx-c50-reducts': Path.cwd() / 'results' / 'multiclassMAECVX' / 'relabelling-reducts-cov50',
    'relabel-msecvx-i50-reducts': Path.cwd() / 'results' / 'multiclassMSECVX' / 'relabelling-reducts-in50',
    'relabel-maecvx-i50-reducts': Path.cwd() / 'results' / 'multiclassMAECVX' / 'relabelling-reducts-in50',
    'relabel-msecvx-i95-reducts': Path.cwd() / 'results' / 'multiclassMSECVX' / 'relabelling-reducts-in95',
    'msecvx-relab-red-cov50-i50-cer51': Path.cwd() / 'results' / 'multiclassMSECVX' / 'relab-red-cov50-inc95-cer51',
    'msecvx-relab-red-cov50-i50-cer55': Path.cwd() / 'results' / 'multiclassMSECVX' / 'relab-red-cov50-inc95-cer55',
    'msecvx-relab-red-cov50-i50-cer60': Path.cwd() / 'results' / 'multiclassMSECVX' / 'relab-red-cov50-inc95-cer60',
    'msecvx-relab-red-cov50-i50-cer70': Path.cwd() / 'results' / 'multiclassMSECVX' / 'relab-red-cov50-inc95-cer70',
    'msecvx-relab-red-cov50-i50-cer51-na': Path.cwd() / 'results' / 'multiclassMSECVX' / 'relab-red-cov50-inc95-cer51-no_add',
    'msecvx-relab-red-cov50-i50-cer55-na': Path.cwd() / 'results' / 'multiclassMSECVX' / 'relab-red-cov50-inc95-cer55-no_add',
    'msecvx-relab-red-cov50-i50-cer60-na': Path.cwd() / 'results' / 'multiclassMSECVX' / 'relab-red-cov50-inc95-cer60-no_add',
    'msecvx-relab-red-cov50-i50-cer70-na': Path.cwd() / 'results' / 'multiclassMSECVX' / 'relab-red-cov50-inc95-cer70-no_add',
    'msecvx-relab02-red-cov50-i50': Path.cwd() / 'results' / 'multiclassMSECVX' / 'relab02-reducts-cov50-inc95',
    'msecvx-relab05-red-cov50-i50': Path.cwd() / 'results' / 'multiclassMSECVX' / 'relab05-reducts-cov50-inc95',
    'msecvx-relab10-red-cov50-i50': Path.cwd() / 'results' / 'multiclassMSECVX' / 'relab10-reducts-cov50-inc95',
    'msecvx-relab15-red-cov50-i50': Path.cwd() / 'results' / 'multiclassMSECVX' / 'relab15-red-cov50-inc95',
    'relabel-maecvx-i95-reducts': Path.cwd() / 'results' / 'multiclassMAECVX' / 'relabelling-reducts-in95',
    'relabel-msecvx-i50+c50-reducts': Path.cwd() / 'results' / 'multiclassMSECVX' / 'relabelling-reducts-incl5-cov05-v2',
    'relabel-maecvx-i50+c50-reducts': Path.cwd() / 'results' / 'multiclassMAECVX' / 'relabelling-reducts-cov50-inc50',
    'relabel-msecvx-i95+c50-reducts': Path.cwd() / 'results' / 'multiclassMSECVX' / 'relabelling-reducts-in95-cov50',
    'relabel-maecvx-i95+c50-reducts': Path.cwd() / 'results' / 'multiclassMAECVX' / 'relabelling-reducts-cov50-inc95',
    # 'base-no-reducts': Path.cwd() / results_base / 'inclusion' / 'no-reducts',
    # 'owa-lower-base': Path.cwd() / results_base / 'owa-lower' / 'default',
    # 'default-check': Path.cwd() / 'results' / 'default',
    # 'prio 1e-6' : Path.cwd() / 'results' / 'lower-approx' / 'prior-inclusion' / '1e-6',
    # 'prio-max 1e-6' : Path.cwd() / 'results' / 'lower-approx' / 'prior-inclusion' / '1e-6-scale',
    # '1e-6-high' : Path.cwd() / 'results' / 'lower-approx' / 'prior-inclusion' / '1e-6-high',
    # '1e-6-high-sum' : Path.cwd() / 'results' / 'lower-approx' / 'prior-inclusion' / '1e-6-high-sum',
    # '1e-6-trueprior-influence-1e-2': Path.cwd() / 'results' / 'lower-approx' / 'prior-inclusion' / '1e-6-trueprior-influence-1e-2',
    # '1e-6-trueprior-influence-1e-3' : Path.cwd() / 'results' / 'lower-approx' / 'prior-inclusion' / '1e-6-trueprior-influence-1e-3',
    # '1e-6-trueprior-influence-1e-4' : Path.cwd() / 'results' / 'lower-approx' / 'prior-inclusion' / '1e-6-trueprior-influence-1e-4',
    # '1e-6-trueprior-influence-1e-6' : Path.cwd() / 'results' / 'lower-approx' / 'prior-inclusion' / '1e-6-trueprior-influence-1e-6',
    # '1e-6-scaled_prior-influence-1e-2' : Path.cwd() / 'results' / 'lower-approx' / 'prior-inclusion' / '1e-6-scaled_prior-influence-1e-2',
    # '1e-6-scaled_prior-influence-3e-2': Path.cwd() / 'results' / 'lower-approx' / 'prior-inclusion' / '1e-6-scaled_prior-influence-3e-2',

}

In [273]:
for model, path in frri_models.items():
    print(model)
    scores[model] = calculate_score(
        data_folder=Path.cwd() / 'keel-data',
        results_folder=path, #'rule_prune-rel-0dot00-std',  #  'rel-0dot05-std', #' no-prune-results'/ 'close-min-max-combo-results',  # /
        metric=metric,
        include=data_sets,
        label_encoding=True
    )
    rules[model] = count_all_rules(
        path,
        include=data_sets
    )
    attributes[model] = count_all_attributes(
        path,
        include=data_sets,
        counter='attribute'
    )

incl 1e-4
msecvx-i50-reducts
relabel-msecvx-reducts-default
relabel-maecvx-reducts-default
relabel-msecvx-c50-reducts
relabel-maecvx-c50-reducts
relabel-msecvx-i50-reducts
relabel-maecvx-i50-reducts
relabel-msecvx-i95-reducts
msecvx-relab-red-cov50-i50-cer51
msecvx-relab-red-cov50-i50-cer55
msecvx-relab-red-cov50-i50-cer60
msecvx-relab-red-cov50-i50-cer70
msecvx-relab-red-cov50-i50-cer51-na
msecvx-relab-red-cov50-i50-cer55-na
msecvx-relab-red-cov50-i50-cer60-na
msecvx-relab-red-cov50-i50-cer70-na
msecvx-relab02-red-cov50-i50
msecvx-relab05-red-cov50-i50
msecvx-relab10-red-cov50-i50
msecvx-relab15-red-cov50-i50
relabel-maecvx-i95-reducts
relabel-msecvx-i50+c50-reducts
relabel-maecvx-i50+c50-reducts
relabel-msecvx-i95+c50-reducts
relabel-maecvx-i95+c50-reducts


|## Checking all of the models

In [274]:
scores.keys()

dict_keys(['modlem', 'furia', 'ripper', 'qr', 'naive-bayes', 'svm', 'tree', 'frnn', 'incl 1e-4', 'relabel-msecvx-reducts-default', 'relabel-maecvx-reducts-default', 'relabel-msecvx-c50-reducts', 'relabel-maecvx-c50-reducts', 'relabel-msecvx-i50-reducts', 'relabel-maecvx-i50-reducts', 'relabel-msecvx-i95-reducts', 'msecvx-relab-red-cov50-i50-cer51', 'msecvx-relab-red-cov50-i50-cer55', 'msecvx-relab-red-cov50-i50-cer60', 'msecvx-relab-red-cov50-i50-cer70', 'msecvx-relab02-red-cov50-i50', 'msecvx-relab05-red-cov50-i50', 'msecvx-relab10-red-cov50-i50', 'msecvx-relab15-red-cov50-i50', 'relabel-maecvx-i95-reducts', 'relabel-msecvx-i50+c50-reducts', 'relabel-maecvx-i50+c50-reducts', 'relabel-msecvx-i95+c50-reducts', 'relabel-maecvx-i95+c50-reducts', 'msecvx-i50-reducts', 'msecvx-relab-red-cov50-i50-cer51-na', 'msecvx-relab-red-cov50-i50-cer55-na', 'msecvx-relab-red-cov50-i50-cer60-na', 'msecvx-relab-red-cov50-i50-cer70-na'])

## Tables
set 1 = QR + OWA-variants without pruning
set 2 = focus on pruning

In [275]:
names = {
    1: [
        'qr',
        'lin-owa',
        'trun-lin-owa',
        'trun-exp-owa',
        'modlem'
    ],
    2: [
        'qr',
        'lin-owa',
        'prune-qr',
        'prune-lin-owa'
    ],
    3: [
        'qr',
        'lin-owa',
        'avg-qr',
        'avg-lin-owa'
    ],
    4: [
        'lin-owa',
        'modlem'
    ],
    5: [
        'lin-owa',
        'frnn',
        'svm',
        'naive-bayes',
        'tree',
    ],
    'frri-paper': [
        # 'nori',
        # 'lower-nori',
        # 'lower-check',
        'lower-new-check',
        # 'lower-incl-t1',
        'modlem',
        'qr',
        'furia',
        'ripper',
    ],
    'inclusion' : [
        'incl 1e-6',
        'incl 1e-4',
        # 'incl 1e-3', 
        # 'incl 1e-2', 
        # 'incl 0.95',
        # 'owa 1e-6',
        # 'quickreduct 1e-6',
        'cv-thres',
        # 'frfs',
        # 'prio 1e-6',
        # 'prio-max 1e-6',
        '1e-6-high',
        # '1e-6-high-sum',
        # '1e-6-trueprior-influence-1e-2',
        # '1e-6-trueprior-influence-1e-4',
        '1e-6-scaled_prior-influence-1e-2',
        '1e-6-scaled_prior-influence-3e-2',
    ],
    'quickreduct' : [
        'incl 1e-6',
        'quickreduct 1e-6',
        'frfs'
    ],
    'gran' : [
        'base-no-reducts',
        'msecvx-no-reducts',
        'maecvx-no-reducts',
        'msecvx-0.5',
    ],
    'gran-reducts': [
        'incl 1e-4',
        # 'msecvx-0.5-reducts',
        # 'maecvx-0.5-reducts',
        # 'msecvx-i95-reducts',
        'msecvx-i50-reducts',
        'relabel-maecvx-reducts-default',
        'relabel-maecvx-c50-reducts',
        'relabel-maecvx-i50-reducts',
        'relabel-maecvx-i95-reducts',
        'relabel-maecvx-i50+c50-reducts',
        'relabel-maecvx-i95+c50-reducts',
        'relabel-msecvx-reducts-default',
        'relabel-msecvx-c50-reducts',
        'relabel-msecvx-i50-reducts',
        'relabel-msecvx-i95-reducts',
        'relabel-msecvx-i50+c50-reducts',
        'relabel-msecvx-i95+c50-reducts',

    ],
    'gran-reducts-extra-params' : [
        'incl 1e-4',
        'relabel-msecvx-i50+c50-reducts',
        # 'msecvx-relab-red-cov50-i50-cer50',
        # 'msecvx-relab-red-cov50-i50-cer55',
        # 'msecvx-relab-red-cov50-i50-cer60',
        'msecvx-relab02-red-cov50-i50',
        'msecvx-relab05-red-cov50-i50',
        'msecvx-relab10-red-cov50-i50', 
        'msecvx-relab15-red-cov50-i50', 
    ],
    'gran-certainty' : [
        # 'incl 1e-4',
        'relabel-msecvx-i50+c50-reducts',
        'msecvx-relab-red-cov50-i50-cer51',
        'msecvx-relab-red-cov50-i50-cer55',
        'msecvx-relab-red-cov50-i50-cer60',
        'msecvx-relab-red-cov50-i50-cer70',
    ],
    'gran-certainty-na' : [
        # 'incl 1e-4',
        'relabel-msecvx-i50+c50-reducts',
        'msecvx-relab-red-cov50-i50-cer51-na',
        'msecvx-relab-red-cov50-i50-cer55-na',
        'msecvx-relab-red-cov50-i50-cer60-na',
        'msecvx-relab-red-cov50-i50-cer70-na',
    ],
    'owa' : [
        'incl 1e-4',
        'owa-lower-base'
    ],
    'default-check': [ # -> OK
        'incl 1e-4',
        'default-check'
    ]
}

In [280]:
nr = 'gran-certainty-na'  # 6
save = False

In [281]:
main_folder = nr  # + '-relabel'  # + "small"  # 'tables_inclusion_set2' # 'balanced_acc_incl'  # 'normal_acc'
tables_path_base = Path.cwd() / 'tables' / main_folder

In [282]:
table_acc = pd.DataFrame(index=data_sets, columns=names[nr])

In [283]:
for model in names[nr]:
    for data_set in data_sets:
    # for data_set, score in scores[model].items():
        table_acc.loc[data_set, model] = scores[model][data_set]
# table_acc['max'] = table_acc.max(axis='columns', skipna=True)
table_acc.drop(['glass', 'yeast'], inplace=True)
table_acc.loc['mean'] = table_acc.mean(axis='rows', skipna=True)
table_acc

,relabel-msecvx-i50+c50-reducts,msecvx-relab-red-cov50-i50-cer51-na,msecvx-relab-red-cov50-i50-cer55-na,msecvx-relab-red-cov50-i50-cer60-na,msecvx-relab-red-cov50-i50-cer70-na
australian,0.856478,0.858091,0.854778,0.85353,0.85326
bands,0.662985,0.650194,0.624597,0.566197,0.530673
bupa,0.532221,0.519483,0.507133,0.504167,0.513846
cleveland,0.28676,0.287936,0.294415,0.311782,0.286644
ecoli,0.705842,0.706063,0.707236,0.7368,0.506809
heart,0.739167,0.771667,0.768333,0.770833,0.7825
ionosphere,0.881783,0.881783,0.889755,0.882482,0.862511
pima,0.59648,0.597551,0.600607,0.561445,0.510111
sonar,0.744394,0.744394,0.743485,0.734394,0.73298
spectfheart,0.5,0.5,0.5,0.5,0.5


In [284]:
if save:
    bolded_first = table_acc.apply(lambda data: bold(data), axis=1)
    bolded_first.to_latex(buf= tables_path_base / f'tab{nr}acc.txt', escape=False)

In [285]:
table_rule = pd.DataFrame(index=data_sets, columns=names[nr])
for model in names[nr]:
    # for data_set, rule_count in rules[model].items():
    for data_set in data_sets:
        table_rule.loc[data_set, model] = rules[model][data_set]  #  rule_count
# table_rule.drop(['glass', 'yeast'], inplace=True)
table_rule.loc['mean'] = table_rule.mean(axis='rows', skipna=True)
table_rule

,relabel-msecvx-i50+c50-reducts,msecvx-relab-red-cov50-i50-cer51-na,msecvx-relab-red-cov50-i50-cer55-na,msecvx-relab-red-cov50-i50-cer60-na,msecvx-relab-red-cov50-i50-cer70-na
australian,48.9,47.9,43.4,37.4,28.2
bands,75.1,71.6,53.1,34.2,28.7
bupa,38.4,31.8,19.7,16.3,8.8
cleveland,96.2,96.0,94.4,85.1,50.5
ecoli,52.7,49.6,35.7,26.8,13.6
glass,57.0,55.1,35.9,17.9,7.4
heart,33.7,37.3,35.9,34.7,28.8
ionosphere,22.8,22.5,20.8,21.3,22.6
pima,74.6,68.8,48.5,32.2,25.4
sonar,28.0,28.0,28.0,28.1,29.3


In [286]:
if save:    
    bolded_first = table_rule.apply(lambda data: bold(data, optimum='min', format_string="%.0f"), axis=1)
    bolded_first.to_latex(buf= tables_path_base / f'tab{nr}rules.txt', escape=False)

In [287]:
table_attribute = pd.DataFrame(index=data_sets, columns=names[nr])
for model in names[nr]:
    # for data_set, attribute_count in attributes[model].items():
    for data_set in data_sets:
        table_attribute.loc[data_set, model] = attributes[model][data_set]  # attribute_count
table_attribute.drop(['glass', 'yeast'], inplace=True)
table_attribute.loc['mean'] = table_attribute.mean(axis='rows', skipna=True)
table_attribute

,relabel-msecvx-i50+c50-reducts,msecvx-relab-red-cov50-i50-cer51-na,msecvx-relab-red-cov50-i50-cer55-na,msecvx-relab-red-cov50-i50-cer60-na,msecvx-relab-red-cov50-i50-cer70-na
australian,4.077074,4.130481,4.303307,4.406049,4.070718
bands,3.94959,3.992588,4.270488,4.792876,5.403744
bupa,2.724225,2.870981,2.955199,3.032572,2.334444
cleveland,4.495994,4.50442,4.51844,4.531518,4.356196
ecoli,2.916294,2.979844,3.171519,3.149743,3.272802
heart,3.576542,3.981785,4.006312,4.004366,3.928093
ionosphere,3.611969,3.634637,3.843666,3.91185,4.067437
pima,3.263146,3.400508,3.661137,3.944115,4.419684
sonar,5.855025,5.855025,5.912081,5.93208,5.948361
spectfheart,1.25,1.25,0.9,0.9,0.9


In [288]:
if save:
    bolded_first = table_attribute.apply(lambda data: bold(data, optimum='min', format_string="%.2f"), axis=1)
    bolded_first.to_latex(buf= tables_path_base / f'tab{nr}atts.txt', escape=False)

## Statistical testing

In [246]:
from scipy import stats
import scikit_posthocs as sph

In [268]:
subject = table_acc

In [269]:
no_mean =  subject.drop(labels = 'mean', axis = 'index')
# no_mean.drop('max', axis='columns', inplace=True)

In [270]:
no_mean

,incl 1e-4,msecvx-i50-reducts,relabel-maecvx-reducts-default,relabel-maecvx-c50-reducts,relabel-maecvx-i50-reducts,relabel-maecvx-i95-reducts,relabel-maecvx-i50+c50-reducts,relabel-maecvx-i95+c50-reducts,relabel-msecvx-reducts-default,relabel-msecvx-c50-reducts,relabel-msecvx-i50-reducts,relabel-msecvx-i95-reducts,relabel-msecvx-i50+c50-reducts,relabel-msecvx-i95+c50-reducts
australian,0.787517,0.842375,0.850373,0.850373,0.569359,0.859477,0.86088,0.850373,0.840681,0.844865,0.502179,0.800638,0.856478,0.847267
bands,0.675141,0.6741,0.523886,0.523886,0.5,0.5,0.523886,0.523886,0.530437,0.542326,0.5,0.530437,0.662985,0.542326
bupa,0.600724,0.526257,0.4975,0.5,0.5,0.4975,0.5,0.5,0.495208,0.519359,0.503846,0.495208,0.532221,0.519359
cleveland,0.290397,0.300831,0.312973,0.266758,0.237525,0.258694,0.231389,0.265403,0.291022,0.269662,0.212064,0.276536,0.28676,0.277877
ecoli,0.720659,0.695693,0.247619,0.529192,0.17381,0.247619,0.502696,0.529192,0.247619,0.714563,0.17381,0.247619,0.705842,0.714563
heart,0.7675,0.725,0.780833,0.780833,0.581667,0.7675,0.618333,0.780833,0.743333,0.779167,0.5,0.759167,0.739167,0.786667
ionosphere,0.912468,0.893239,0.671154,0.671154,0.5,0.651282,0.790355,0.671154,0.633974,0.798077,0.514312,0.633974,0.881783,0.798077
pima,0.687633,0.634845,0.5,0.5,0.5,0.5,0.5,0.5,0.526771,0.543993,0.5,0.526771,0.59648,0.543993
sonar,0.786212,0.744394,0.5,0.567222,0.5,0.5,0.724167,0.567222,0.492929,0.786086,0.5,0.492929,0.744394,0.786086
spectfheart,0.628874,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.490584,0.5,0.497727,0.48355,0.5,0.5


In [271]:
stats.wilcoxon(no_mean['relabel-msecvx-i50+c50-reducts'],no_mean['incl 1e-4'], alternative='less')

WilcoxonResult(statistic=23.0, pvalue=0.0635986328125)

In [250]:
f_test = stats.friedmanchisquare(*no_mean.values)
f_test

FriedmanchisquareResult(statistic=110.59030837004401, pvalue=4.573115915610621e-18)

In [129]:
post_hocs = sph.posthoc_conover_friedman(no_mean.values, p_adjust='Holm')
post_hocs.columns=no_mean.columns
post_hocs.index=no_mean.columns
post_hocs

,incl 1e-6,incl 1e-4,incl 1e-3,incl 1e-2,cv-thres
incl 1e-6,1.000000,1.0,1.0,0.757743,0.858925
incl 1e-4,1.000000,1.0,1.0,1.000000,1.000000
incl 1e-3,1.000000,1.0,1.0,1.000000,1.000000
incl 1e-2,0.757743,1.0,1.0,1.000000,1.000000
cv-thres,0.858925,1.0,1.0,1.000000,1.000000


In [65]:
best = no_mean.max(axis='columns')
dif = no_mean.subtract(best, axis='rows').abs()
dif.max(axis='rows').sort_values()

incl 1e-3     0.04599
incl 1e-6    0.047052
cv-thres     0.050783
incl 1e-4    0.050873
incl 1e-2    0.181944
dtype: object

In [189]:
ranks = no_mean.rank(axis='columns', ascending=False)
friedman_ranks = ranks.mean()
print(friedman_ranks.sort_values().to_latex(escape=False))

\begin{tabular}{lr}
\toprule
{} &         0 \\
\midrule
incl 1e-6 &  2.433333 \\
incl 1e-4 &  2.600000 \\
incl 1e-3 &  2.900000 \\
cv-thres  &  3.433333 \\
incl 1e-2 &  3.633333 \\
\bottomrule
\end{tabular}


/var/folders/_f/yf3dkr5x1397v4ntwhncxfhw0000gn/T/ipykernel_53128/2662355144.py:3: FutureWarning: In future versions `DataFrame.to_latex` is expected to utilise the base implementation of `Styler.to_latex` for formatting and rendering. The arguments signature may therefore change. It is recommended instead to use `DataFrame.style.to_latex` which also contains additional functionality.
  print(friedman_ranks.sort_values().to_latex(escape=False))


In [67]:
ranks['lower-new-check'].value_counts()

KeyError: 'lower-new-check'

In [68]:
ranks.apply(pd.value_counts)

,incl 1e-6,incl 1e-4,incl 1e-3,incl 1e-2,cv-thres
1.0,3,5.0,1.0,3,2
2.0,4,3.0,3.0,1,2
2.5,1,NaN,1.0,1,1
3.0,5,1.0,7.0,1,2
4.0,1,1.0,3.0,3,5
5.0,1,5.0,NaN,6,3


In [188]:
median = no_mean.median(axis='rows')
median

incl 1e-6    0.680919
incl 1e-4    0.687633
incl 1e-3    0.676793
incl 1e-2    0.660827
cv-thres     0.668552
dtype: float64

On high IR datasets (>6)

In [196]:
ir_datasets = [
    'cleveland',
    'ecoli',
    'glass',
    'spectfheart',
    'winequality-red',
    'yeast'
]

ir_no_mean = no_mean.loc[ir_datasets]
# ir_no_mean.drop('qr', axis='columns', inplace=True)

In [197]:
f_test = stats.friedmanchisquare(*ir_no_mean.values)
f_test

FriedmanchisquareResult(statistic=14.714285714285708, pvalue=0.011655524770004939)

In [198]:
post_hocs = sph.posthoc_conover_friedman(ir_no_mean.values)
post_hocs.columns=ir_no_mean.columns
post_hocs.index=ir_no_mean.columns
post_hocs

,lower-new-check,modlem,qr,furia,ripper
lower-new-check,1.000000,0.233052,0.004900,0.094249,0.094249
modlem,0.233052,1.000000,0.067585,0.603959,0.603959
qr,0.004900,0.067585,1.000000,0.175230,0.175230
furia,0.094249,0.603959,0.175230,1.000000,1.000000
ripper,0.094249,0.603959,0.175230,1.000000,1.000000


In [199]:
stats.wilcoxon(ir_no_mean['lower-new-check'], ir_no_mean['ripper'], alternative="greater")

WilcoxonResult(statistic=18.0, pvalue=0.078125)

In [200]:
wil_ph = sph.posthoc_nemenyi_friedman(ir_no_mean.values)
wil_ph.columns=ir_no_mean.columns
wil_ph.index=ir_no_mean.columns
wil_ph

,lower-new-check,modlem,qr,furia,ripper
lower-new-check,1.000000,0.680037,0.008987,0.359170,0.359170
modlem,0.680037,1.000000,0.261866,0.900000,0.900000
qr,0.008987,0.261866,1.000000,0.576422,0.576422
furia,0.359170,0.900000,0.576422,1.000000,0.900000
ripper,0.359170,0.900000,0.576422,0.900000,1.000000


0.03